# 网络集合## 简介[NetworkSet](../api/networkSet.rst) 对象表示一个无序的网络集合。它提供了迭代和切片集合的方法，按日期时间排序，计算统计量，并在图表中显示不确定度范围。## 创建 [NetworkSet](../api/networkSet.rst)

让我们看一下 `data/` 文件夹，里面有一些名为 `ro` 的网络的重复测量数据，这是一个*辐射开路*波导。

```ls data/ro*-a----       14/02/2021     12:35           8031 ro,1.s1p-a----       14/02/2021     12:35           8030 ro,2.s1p-a----       14/02/2021     12:35           8031 ro,3.s1p-a----       14/02/2021     12:35          46592 ro_spreadsheet.xls```

文件 `ro,1.s1p`、`ro,2.s1p` 等是重复的测量数据，我们希望使用 [NetworkSet](../api/networkSet.rst) 类来计算这些数据的统计信息。[NetworkSet](../api/networkSet.rst) 是由 [Network](../api/network.rst) 对象的列表或字典创建的。因此，首先我们需要将所有 touchstone 文件加载到 `Networks` 中。这可以使用 `rf.io.read_all` 快速完成。`contains` 参数用于仅加载与给定子字符串匹配的文件。

In [ ]:
import skrf as rf

rf.io.read_all(rf.data.pwd, contains='ro')

这可以直接传递给 `[NetworkSet](../api/networkSet.rst)` 构造函数。

In [ ]:
from skrf.networkSet import NetworkSet

ro_dict = rf.io.read_all(rf.data.pwd, contains='ro')
ro_ns = NetworkSet(ro_dict, name='ro set')
ro_ns

也可以直接从以下内容构建 NetworkSet：- 包含 Touchstone 文件的目录：`NetworkSet.from_dir()`，- Touchstone 文件的 zip 文件：`NetworkSet.from_zip()`，- 散射参数字典：`NetworkSet.from_s_dict()`，- (G)MDIF (.mdf) 文件：`NetworkSet.from_mdif()`，- CITI (.cti) 文件：`NetworkSet.from_citi()`。

## 访问网络方法在 [NetworkSet](../api/networkSet.rst) 中，可以通过类似于访问列表元素的方式来访问 [Network](../api/network.rst) 元素。

In [ ]:
ro_ns[0]

大多数 [Network](../api/network.rst) 方法也是 [NetworkSet](../api/networkSet.rst) 的方法。这些方法会分别应用于每个 [Network](../api/network.rst) 元素。例如，要绘制每个 Network 的 s 参数的对数幅度。

In [ ]:
%matplotlib inline
import skrf as rf

rf.stylely()

ro_ns.plot_s_db()

## 统计特性可以通过访问 NetworkSet 的属性来计算统计量。要计算集合的复数平均值，请访问 `mean_s` 属性。

In [ ]:
ro_ns.mean_s

统计算子的命名约定为 `NetworkSet.{function}_{parameter}`，其中 `function` 是统计函数的名称，`parameter` 是要进行运算的网络参数。这些方法返回一个 [Network](../api/network.rst) 对象，因此可以像处理普通 Network 对象一样对其进行保存或绘制。要绘制复数平均响应的对数幅度，

In [ ]:
ro_ns.mean_s.plot_s_db(label='ro')

或者绘制复数散射参数的标准偏差。

In [ ]:
ro_ns.std_s.plot_s_re(y_label='Standard Deviations')

使用这些属性，可以计算复数网络参数的标量分量的统计量。要计算相位分量的平均值，

In [ ]:
ro_ns.mean_s_deg.plot_s_re()

## 绘制不确定度范围可以通过以下方法绘制不确定度范围。

In [ ]:
ro_ns.plot_uncertainty_bounds_s_db()

In [ ]:
ro_ns.plot_uncertainty_bounds_s_deg()

## 绘制小提琴图小提琴图用于快速了解数据的分布情况，它通过在每个点显示概率密度，以及默认情况下显示最小值、最大值和平均值来实现。让我们对频率进行插值，以减少图表中的杂乱信息。

In [ ]:
ro_ns_interp = ro_ns.interpolate_frequency(rf.Frequency(500, 600, 15, "GHz"))

对于时域中的参数以外的大多数常见网络参数，都可以绘制小提琴图。要绘制散射参数的对数幅值和相位，请执行以下操作。

In [ ]:
ro_ns_interp.plot_violin("s_db")

In [ ]:
ro_ns_interp.plot_violin("s_deg")

## 读取和写入

将 [NetworkSet](../api/networkSet.rst) 中的所有 [Network](../api/network.rst) 数据分别写入单独的触点文件中。

In [ ]:
ro_ns.write_touchstone(dir='data/')

为了临时存储数据，可以使用 `rf.io.read` 和 `rf.io.write` 函数将 [NetworkSet](../api/networkSet.rst) 保存到磁盘并从磁盘读取。

In [ ]:
rf.io.write('ro set.ns', ro_ns)

In [ ]:
ro_ns = rf.io.read('ro set.ns')
ro_ns

## 导出为 Excel、CSV 或 HTML 格式

[NetworkSet](../api/networkSet.rst) 也可以导出为其他文件类型。输出格式（实部/虚部、幅值/相位）和输出类型（csv、excel、html）均可调整。例如，可以将每个网络的幅值/相位导出到 Excel 表格中，以便向您的老板[们]展示。

In [ ]:
ro_ns.write_spreadsheet('data/ro_spreadsheet.xls', form='db')

有关此功能的更多信息，请参阅函数 `skrf.io.general.network_2_spreadsheet`。

## 命名参数如果 `NetworkSet` 中的所有 `Network` 对象都具有一个 `params` 属性，该属性包含一个字典，其中包含与每个 `Network` 关联的命名参数及其值，则可以使用 `.sel()` 方法选择与命名参数子集对应的 `Network`。以下示例说明了此功能。

In [ ]:
# dummy named parameters and values 'a', 'X' and 'c'
import numpy as np

params = [
        {'a':0, 'X':10, 'c':'A'},
        {'a':1, 'X':10, 'c':'A'},
        {'a':2, 'X':10, 'c':'A'},
        {'a':1, 'X':20, 'c':'A'},
        {'a':0, 'X':20, 'c':'A'},
        ]
# create a NetworkSet made of dummy Networks, each define for set of parameters
freq1 = rf.Frequency(75, 110, 101, 'ghz')
rng = np.random.default_rng()
ntwks_params = [rf.Network(frequency=freq1, s=rng.uniform(size=(len(freq1),2,2)),
                               name=f'ntwk_{m}', comment=f'ntwk_{m}', params=params)
                            for (m, params) in enumerate(params) ]
ns = rf.networkSet.NetworkSet(ntwks_params)
print(ns)

可以选择与标量参数匹配的子 NetworkSet，具体方法如下：

In [ ]:
ns.sel({'a': 1})

In [ ]:
ns.sel({'a': 0, 'X': 10})

选择与一系列参数范围匹配的子 NetworkSet 同样有效：

In [ ]:
ns.sel({'a': 0, 'X': [10,20]})

In [ ]:
ns.sel({'a': [0,1], 'X': [10,20]})

可以使用 `dims` 和 `coords` 属性来检索 `NetworkSet` 中各种命名的参数键和值：

In [ ]:
ns.dims

In [ ]:
ns.coords

## 在网络集中的网络之间进行插值可以从`NetworkSet`中包含的网络中插值生成新的网络。如果`NetworkSet`中的每个网络都没有定义`params`属性，则可以使用`interpolate_from_network()`方法来指定要插值的参数。

In [ ]:
param_x = [1, 2, 3]  # a parameter associated to each Network
x0 = 1.5  # parameter value to interpolate for
interp_ntwk = ro_ns.interpolate_from_network(param_x, x0)
print(interp_ntwk)

文档的[示例部分](../examples/index.rst#networksets)中提供了一个图示示例。当参数已定义时，也可以使用命名参数进行插值：

In [ ]:
# Interpolated Network for a=1.2 within X=10 Networks:
ns.interpolate_from_params('a', 1.2, {'X': 10})